In [2]:
# 此脚本用于合并exoRbase来源全体序列和microarray来源根据序列，microarray序列通过细胞内表达量进行下采样
import pandas as pd
from pyfaidx import Fasta
from Bio import SeqIO
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from Bio.Seq import Seq
import sys

In [ ]:
exoRbase_df = pd.read_csv('./sequence_camparison/outputs/exoRbase_filtered.csv')
microarrary_df = pd.read_csv('./sequence_camparison/outputs/celline_filtered.csv')
exoRbase_df

,Unnamed: 0,circID,matched_circAltas_IDs,sum_EVP,Sequence,Sequence_Length
0,0,exo_circ_000216,hsa-MTATP6P1_0003,1,GTGTGCCTTGTGGTAAGAAGTGGGCTAGGGCATTTTTAATCTTAGA...,302
1,1,exo_circ_000253,hsa-MTATP6P1_0009,18,GTGTGCCTTGTGGTAAGAAGTGGGCTAGGGCATTTTTAATCTTAGA...,200
2,2,exo_circ_000254,hsa-MTATP6P1_0002,226,GGGTGTAGGTGTGCCTTGTGGTAAGAAGTGGGCTAGGGCATTTTTA...,208
3,3,exo_circ_000310,hsa-MTATP6P1_0011,1,GCTGATGGTTTCGATAATAACTAGTATAGGGATAAGGGGTGTAGGT...,164
4,4,exo_circ_000327,hsa-MTATP6P1_0012,3,GCTGATGGTTTCGATAATAACTAGTATAGGGATAAGGGGTGTAGGT...,157
...,...,...,...,...,...,...
75176,75564,exo_circ_126503,hsa-TMLHE-AS1_0024,15,ATCAGTTAAGCAAAGGAAACAGAAGTACATGTGTATGTACAC,42
75177,75565,exo_circ_126592,hsa-ZFY_0001,21,GAGCTGTGACTAATGAGAATTAAAGGCCATGGATGAAGATGAATTT...,662
75178,75566,exo_circ_126758,hsa-TTTY14_0003,8,AACTGTCCATTTGGCTATGAGCGAATCATCCTGCTTGCTTTTATAA...,431
75179,75567,exo_circ_126785,hsa-TXLNGY_0002,17,TTTGTGAATAGCACAATGGAAGAAGCTGGACTTTGTGGGTTAAGAG...,298


In [4]:
microarrary_df

,CircRNA,MDA-MB-231_original,Spliced seq length,Sequence
0,hsa_circ_0009688,1014.974000,229,TATTCTCCGATTTTAAGGACTTGATTGGCCAGATTTTAATGGAAGT...
1,hsa_circ_0112904,2095.288000,572,TACATTTTTATACCTGAAATTCCTGGTGGTGTGGGCACTTGTCCTC...
2,hsa_circ_0015377,743.614600,197,TGAATGAACTCATTCTTAAACAGAAGCAAAGATTTGAGGAAAAGAG...
3,hsa_circ_0113417,2930.645186,571,CCTGGCCAAGCACCGAATAGGGACAAAGAGGCACCGAGTTTGTCTT...
4,hsa_circ_0113162,1286.246122,26899,ACATTTTAAATCCAAAGGATGTGATCAGTGCCCAGTTTGAAAACAC...
...,...,...,...,...
110131,hsa_circ_0006322,169.901100,441,GACTTGGACCCTCTGTGGAACACCCGAGTACCTAGCCCCCGAAGTC...
110132,hsa_circ_0092258,1277.163000,186,GTGTGGCAGAAAAAACACAGCTTCTGAAATTGAATGTACCTGCTAC...
110133,hsa_circ_0092255,198.896600,444,GAATTAATTGATGATTTCATCTTTCCCGCATCCAAAGTTTACCTGC...
110134,hsa_circ_0092207,2798.053000,1924,TGCCTGCTACTCACCCCACGCAGCCTACGCCCAGAGCAAGCTGGCC...


In [5]:
microarrary_df = microarrary_df.sort_values(by='MDA-MB-231_original', ascending=False).head(75569)

# 查看处理后的 dataframe
microarrary_df

,CircRNA,MDA-MB-231_original,Spliced seq length,Sequence
72551,hsa_circ_0067964,165729.300000,283,GCGGCGGCGGCGGCTGGAGGAGGAGAGCGGCGGCGGCGGGAGCAGC...
45668,hsa_circ_0041846,151361.800000,2378,AACTGCTGTCGCCGCCGCCGCCGCCGCCTCAGCTTCCCACAGCCGC...
48403,hsa_circ_0107190,147917.900000,234,GTGAAAAGAGAACAAGCGAAAGATGGATGAAGTGGCGGGGAGGAGA...
73611,hsa_circ_0067965,147466.100000,359,GCGGCGGCGGCGGCTGGAGGAGGAGAGCGGCGGCGGCGGGAGCAGC...
103896,hsa_circ_0138219,131773.500000,318,GTGAGCGGGCTGTGCTGGGTGGGGTGTGTGTGACTGTGTGTGTAAG...
...,...,...,...,...
36103,hsa_circ_0037060,462.791400,292,CTTGGGCATGAACCACGACGATGACCACTCATCTTGCGCTGGCAGG...
49323,hsa_circ_0047127,462.771565,487,TGGTCAAGTGGGTTCTCGGAGTCCTTCTATGATTAGTAATGATTCT...
45388,hsa_circ_0042954,462.723900,182,CCTTCAGGCTCTACACACAGATGATCCTCTCACTTGGGATTATGTG...
97773,hsa_circ_0081891,462.706400,262,TCCAGAAAAGGCTTTGAAAGACTCACTACAACCCTATGAGGCTGCT...


In [6]:
def save_merged_to_fasta(micro_df, exo_df, output_filename):
    with open(output_filename, 'w') as f:
        # 处理 microarrary_df，添加 _0 后缀
        for _, row in micro_df.iterrows():
            header = f">{row['CircRNA']}_0"
            # 兼容 Sequence 和 sequence 列名的写法
            seq = str(row['Sequence']) if 'Sequence' in row else str(row['sequence'])
            f.write(f"{header}\n{seq}\n")
            
        # 处理 exoRbase_df，添加 _1 后缀
        for _, row in exo_df.iterrows():
            header = f">{row['circID']}_1"
            seq = str(row['Sequence']) if 'Sequence' in row else str(row['sequence'])
            f.write(f"{header}\n{seq}\n")

# 调用函数保存为合并的 fasta 文件
save_merged_to_fasta(microarrary_df, exoRbase_df, "./merged_dataset.fasta")